# Create Test Dataset for Cost Benchmark
## Creates a 1000 sample test set with:
## - 10% BEGIN samples (100)
## - 80% MIDDLE samples (800)
## - 10% END samples (100)

## Install Packages

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install datasets pandas huggingface_hub
else:
    !pip install datasets pandas huggingface_hub

In [2]:
%%capture
!pip install tiktoken pandas datasets huggingface_hub transformers

## Import Libraries

In [3]:
import pandas as pd
from datasets import Dataset, load_dataset
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
import os
import tiktoken
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
import glob
from tqdm.auto import tqdm

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## Mount Drive

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


## Login to HuggingFace

In [5]:
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in successfully")
else:
    print("No HF_TOKEN found")

Logged in successfully


## Load Full Dataset

In [6]:
csv_path = "/content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv"
df = pd.read_csv(csv_path)

print(f"Total samples in dataset: {len(df)}")
print(f"\nConversation stages:")
print(df['conversation_stage'].value_counts())

Total samples in dataset: 36669

Conversation stages:
conversation_stage
MIDDLE       26125
END           5612
BEGINNING     4932
Name: count, dtype: int64


## Sample Data by Stage
## BEGIN: 10%, MIDDLE: 80%, END: 10%

In [7]:
# Sample 100 from BEGIN
begin_samples = df[df['conversation_stage'] == 'BEGINNING'].sample(n=100, random_state=42)

# Sample 800 from MIDDLE
middle_samples = df[df['conversation_stage'] == 'MIDDLE'].sample(n=800, random_state=42)

# Sample 100 from END
end_samples = df[df['conversation_stage'] == 'END'].sample(n=100, random_state=42)

# Combine all samples
test_df = pd.concat([begin_samples, middle_samples, end_samples], ignore_index=True)

# Shuffle the combined dataset
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Clear generated_answer column
test_df['generated_answer'] = ''

print(f"\nTest dataset created with {len(test_df)} samples")
print(f"BEGIN: {len(begin_samples)}, MIDDLE: {len(middle_samples)}, END: {len(end_samples)}")


Test dataset created with 1000 samples
BEGIN: 100, MIDDLE: 800, END: 100


## Save to Drive

In [8]:
test_dataset_dir = "/content/drive/MyDrive/fyp-2025/Datasets/TestData/cost_benchmark_testset"
os.makedirs(test_dataset_dir, exist_ok=True)

test_csv_path = os.path.join(test_dataset_dir, "cost_benchmark_test_1000.csv")
test_df.to_csv(test_csv_path, index=False)

print(f"Saved to: {test_csv_path}")

Saved to: /content/drive/MyDrive/fyp-2025/Datasets/TestData/cost_benchmark_testset/cost_benchmark_test_1000.csv


## Push to HuggingFace

In [9]:
dataset = Dataset.from_pandas(test_df)
repo_id = "Lakshan2003/slm-cost-benchmark-testset-1000"

dataset.push_to_hub(repo_id, private=True)
print(f"Pushed to HuggingFace: {repo_id}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.03MB / 1.03MB            

README.md:   0%|          | 0.00/582 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to HuggingFace: Lakshan2003/slm-cost-benchmark-testset-1000


## Verify Dataset

In [10]:
print(f"\nDataset Shape: {test_df.shape}")
print(f"\nColumns: {test_df.columns.tolist()}")
print(f"\nFirst sample:")
print(test_df.iloc[0])


Dataset Shape: (1000, 8)

Columns: ['conversation_id', 'instruction', 'history', 'history_summary', 'client_question', 'ground_truth', 'generated_answer', 'conversation_stage']

First sample:
conversation_id                        477e311284084f06a1ede0e8f33c9fce
instruction           You are a professional call-center customer se...
history               Agent: Good morning, thank you for calling Hor...
history_summary       Rex, the client, inquired about making his cha...
client_question       Okay, that makes sense. But what if I'm not su...
ground_truth          I understand that it can be tough to choose th...
generated_answer                                                       
conversation_stage                                               MIDDLE
Name: 0, dtype: object


# Calculate GPT Tokens for All Models
## Adds token count column to each model's results and metrics
## Calculates time per token metrics
## Updates HuggingFace datasets with new columns

## Configuration

In [11]:
BASE_PATH = "/content/drive/MyDrive/fyp-2025/cost_benchmark_results"

# Model configurations with their HuggingFace repos
MODELS = [
    {
        "name": "Llama-3.2-1B-Instruct",
        "repo": "Lakshan2003/Llama-3.2-1B-Instruct-cost-benchmark-results"
    },
    {
        "name": "Qwen3-1.7B-Instruct",
        "repo": "Lakshan2003/Qwen3-1.7B-Instruct-cost-benchmark-results"
    },
    {
        "name": "LLaMA-3.2-3B-Instruct",
        "repo": "Lakshan2003/LLaMA-3.2-3B-Instruct-cost-benchmark-results"
    },
    {
        "name": "SmolLM3-3B-Instruct",
        "repo": "Lakshan2003/SmolLM3-3B-Instruct-cost-benchmark-results"
    },
    {
        "name": "Phi-4-Mini",
        "repo": "Lakshan2003/Phi-4-Mini-cost-benchmark-results"
    },
    {
        "name": "Qwen3-4B-Instruct",
        "repo": "Lakshan2003/Qwen3-4B-Instruct-cost-benchmark-results"
    },
    {
        "name": "Gemma-3-4B-Instruct",
        "repo": "Lakshan2003/Gemma-3-4B-Instruct-cost-benchmark-results"
    },
    {
        "name": "LLaMA-3.1-8B-Instruct",
        "repo": "Lakshan2003/LLaMA-3.1-8B-Instruct-cost-benchmark-results"
    },
    {
        "name": "Qwen3-8B-Instruct",
        "repo": "Lakshan2003/Qwen3-8B-Instruct-cost-benchmark-results"
    }
]

print(f"Will process {len(MODELS)} models")

Will process 9 models


## Setup Tokenizer

In [12]:
import os
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import tiktoken

# - TOKENIZER -
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    if pd.isna(text) or not str(text).strip():
        return 0
    return len(tokenizer.encode(str(text)))

def normalize_columns(df):
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def safe_series(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series([0] * len(df))

## Process Each Model

In [24]:
summary_stats = []

for model_config in tqdm(MODELS, desc="Processing models"):
    model_name = model_config["name"]
    repo_id = model_config["repo"]

    print(f"\n{'='*80}")
    print(f"Processing: {model_name}")
    print(f"{'='*80}")

    model_dir = os.path.join(BASE_PATH, model_name)
    results_csv = os.path.join(model_dir, f"{model_name}_results.csv")
    combined_csv = os.path.join(model_dir, "combined_results_metrics.csv")

    if not os.path.exists(results_csv) or not os.path.exists(combined_csv):
        print(" Missing CSVs → skipping")
        continue

    # - LOAD FILES -
    results_df = normalize_columns(pd.read_csv(results_csv))
    combined_df = normalize_columns(pd.read_csv(combined_csv))

    # - FIND ANSWER COLUMN -
    answer_cols = ["generated_answer", "response", "assistant_response", "model_output"]
    answer_col = next((c for c in answer_cols if c in results_df.columns), None)

    if answer_col and "output_tokens" not in results_df.columns:
        print(f"  Computing tokens from '{answer_col}'")
        results_df["output_tokens"] = results_df[answer_col].apply(count_tokens)
    elif "output_tokens" not in results_df.columns:
        results_df["output_tokens"] = 0

    results_df.to_csv(results_csv, index=False)

    #  MERGE TOKENS SAFELY
    print("  Merging tokens into combined file...")

    token_map = results_df[["conversation_id", "output_tokens"]]

    combined_df = combined_df.merge(
        token_map,
        on="conversation_id",
        how="left",
        suffixes=("", "_new")
    )

    if "output_tokens_new" in combined_df.columns:
        combined_df["output_tokens"] = combined_df["output_tokens_new"].fillna(
            combined_df.get("output_tokens", 0)
        )
        combined_df.drop(columns=["output_tokens_new"], inplace=True)

    if "output_tokens" not in combined_df.columns:
        combined_df["output_tokens"] = 0

    combined_df["output_tokens"] = combined_df["output_tokens"].fillna(0)

    #  ENSURE METRIC COLUMNS
    for col in ["latency_seconds", "ttft_seconds", "gpu_memory_mb", "disk_storage_gb"]:
        if col not in combined_df.columns:
            combined_df[col] = 0

    #  SECONDS PER TOKEN
    combined_df["seconds_per_token"] = combined_df.apply(
        lambda r: r["latency_seconds"] / r["output_tokens"] if r["output_tokens"] > 0 else 0,
        axis=1
    )

    combined_df.to_csv(combined_csv, index=False)

    #  ROBUST SUMMARY
    latency = safe_series(combined_df, "latency_seconds")
    ttft = safe_series(combined_df, "ttft_seconds")
    gpu = safe_series(combined_df, "gpu_memory_mb")
    tokens = safe_series(combined_df, "output_tokens")
    spt = safe_series(combined_df, "seconds_per_token")
    disk = safe_series(combined_df, "disk_storage_gb")

    stats = {
        "model": model_name,
        "samples": len(combined_df),

        "avg_latency_seconds": latency.mean(),
        "median_latency_seconds": latency.median(),
        "std_latency_seconds": latency.std(),

        "avg_ttft_seconds": ttft.mean(),
        "median_ttft_seconds": ttft.median(),

        "avg_gpu_memory_mb": gpu.mean(),
        "max_gpu_memory_mb": gpu.max(),

        "disk_storage_gb": disk.iloc[0] if len(disk) > 0 else 0,

        "avg_output_tokens": tokens.mean(),
        "total_output_tokens": tokens.sum(),

        "avg_seconds_per_token": spt.mean(),
        "median_seconds_per_token": spt.median(),
    }

    # Include cost metrics automatically
    for c in combined_df.columns:
        if "cost" in c.lower() or "usd" in c.lower():
            series = safe_series(combined_df, c)
            stats[f"avg_{c}"] = series.mean()
            stats[f"total_{c}"] = series.sum()

    summary_stats.append(stats)

    #  PUSH TO HF -
    print(f"Pushing data → {repo_id}")
    Dataset.from_pandas(combined_df, preserve_index=False).push_to_hub(repo_id, private=True)

print("\nAll models processed successfully.")
pd.DataFrame(summary_stats).to_csv("all_models_summary.csv", index=False)



Processing models:   0%|          | 0/9 [00:00<?, ?it/s]


Processing: Llama-3.2-1B-Instruct
  Merging tokens into combined file...
Pushing data → Lakshan2003/Llama-3.2-1B-Instruct-cost-benchmark-results


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.16MB / 1.16MB            

                              : 100%|##########| 1.16MB / 1.16MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing models: 100%|██████████| 9/9 [00:19<00:00,  2.12s/it]


All models processed successfully.


## Create Final Summary Table

In [25]:
# Create summary DataFrame
summary_df = pd.DataFrame(summary_stats)

# Round numeric columns
numeric_cols = summary_df.select_dtypes(include=['float64']).columns
summary_df[numeric_cols] = summary_df[numeric_cols].round(4)

# Display summary
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80)
print(summary_df.to_string(index=False))

# Save to CSV
summary_csv_path = os.path.join(BASE_PATH, "final_summary_with_tokens.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nSummary saved to: {summary_csv_path}")


FINAL SUMMARY STATISTICS
                model  samples  avg_latency_seconds  median_latency_seconds  std_latency_seconds  avg_ttft_seconds  median_ttft_seconds  avg_gpu_memory_mb  max_gpu_memory_mb  disk_storage_gb  avg_output_tokens  total_output_tokens  avg_seconds_per_token  median_seconds_per_token
Llama-3.2-1B-Instruct     1000               0.9355                  0.9004               0.2726            0.0245               0.0231          2412.1194          2425.0942           2.3535             44.838                44838                 0.0210                    0.0209
  Qwen3-1.7B-Instruct     1000               1.8255                  1.7594               0.5510            0.0495               0.0481          3995.9958          4031.7598           3.8653             41.045                41045                 0.0446                    0.0443
LLaMA-3.2-3B-Instruct     1000               1.5917                  1.5310               0.4945            0.0383               0.036

In [26]:
import pandas as pd

# Convert MB to GB
summary_df["avg_gpu_memory_gb"] = summary_df["avg_gpu_memory_mb"] / 1024
summary_df["max_gpu_memory_gb"] = summary_df["max_gpu_memory_mb"] / 1024

# Drop old MB columns
summary_df = summary_df.drop(
    columns=["avg_gpu_memory_mb", "max_gpu_memory_mb"]
)

# Reorder columns (optional but clean)
cols = [
    "model",
    "samples",
    "avg_latency_seconds",
    "median_latency_seconds",
    "std_latency_seconds",
    "avg_ttft_seconds",
    "median_ttft_seconds",
    "avg_gpu_memory_gb",
    "max_gpu_memory_gb",
    "disk_storage_gb",
    "avg_output_tokens",
    "total_output_tokens",
    "avg_seconds_per_token",
    "median_seconds_per_token",
]
summary_df = summary_df[cols]

# Round all numeric columns to 2 decimals
summary_df = summary_df.round(2)

# Print final summary
print("\n" + "=" * 80)
print("FINAL SUMMARY STATISTICS")
print("=" * 80)
print(summary_df.to_string(index=False))


FINAL SUMMARY STATISTICS
                model  samples  avg_latency_seconds  median_latency_seconds  std_latency_seconds  avg_ttft_seconds  median_ttft_seconds  avg_gpu_memory_gb  max_gpu_memory_gb  disk_storage_gb  avg_output_tokens  total_output_tokens  avg_seconds_per_token  median_seconds_per_token
Llama-3.2-1B-Instruct     1000                 0.94                    0.90                 0.27              0.02                 0.02               2.36               2.37             2.35              44.84                44838                   0.02                      0.02
  Qwen3-1.7B-Instruct     1000                 1.83                    1.76                 0.55              0.05                 0.05               3.90               3.94             3.87              41.04                41045                   0.04                      0.04
LLaMA-3.2-3B-Instruct     1000                 1.59                    1.53                 0.49              0.04                 0.0

In [27]:
# Push final summary table to HuggingFace
summary_repo = "Lakshan2003/slm-cost-benchmark-final-summary"

print(f"Pushing final summary to HuggingFace: {summary_repo}")
try:
    summary_dataset = Dataset.from_pandas(summary_df)
    summary_dataset.push_to_hub(summary_repo, private=True)
    print(f"Successfully pushed final summary to HuggingFace")
except Exception as e:
    print(f"Error pushing summary: {e}")

Pushing final summary to HuggingFace: Lakshan2003/slm-cost-benchmark-final-summary


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 7.48kB / 7.48kB            

Successfully pushed final summary to HuggingFace
